# Step 1 + Step 2 · 파이프라인 개조 & 모델 베이크오프 (Colab 판)

재활용품 VQA 챌린지 — **Colab GPU / 하루 예산** 기준. 베이스라인 `260827_baseline_colab.ipynb`의 환경 설정을 그대로 따릅니다.

| | 내용 |
|---|---|
| **Step 1** | 전체 데이터 로드 · 자체 val split · 텍스트 전용 기준선 재현 · **a/b/c/d 로짓 비교 추론**으로 교체 |
| **Step 2** | 3개 모델 **zero-shot** 비교 (학습 없음) + 시각화 |

비교 대상: `Qwen3-VL-4B-Instruct` · `kanana-1.5-v-3b-instruct` · `Qwen2.5-VL-3B-Instruct`(베이스라인 대조군)

> **판정 기준**: 이미지를 전혀 안 준 텍스트 전용 분류기가 **55~58%**를 냅니다(3번 셀에서 이 split 기준으로 직접 재계산).
> 어떤 모델이든 이 값을 못 넘으면 이미지를 못 보고 있는 것이고, 학습으로 넘어가면 안 됩니다.

### 데스크탑 판과 달라진 점 (Colab 대응)

| 항목 | 대응 |
|---|---|
| **T4는 bfloat16 미지원** | GPU compute capability를 보고 `fp16`/`bf16`을 자동 선택. 잘못 쓰면 그냥 느려지거나 NaN이 납니다 |
| **세션이 끊긴다** | 모델 1개 끝날 때마다 결과를 **Drive에 즉시 저장**. 재실행하면 끝난 모델은 건너뜁니다 |
| **VRAM 15GB (T4)** | VRAM을 보고 배치 크기 자동 결정 + **OOM이 나면 배치를 반으로 쪼개 재시도** |
| **런타임 초기화 시 파일 소실** | 출력은 전부 `/content` 아닌 **Drive**에 저장 |
| **한글 폰트 없음** | `fonts-nanum` 설치 후 matplotlib에 직접 등록 (런타임 재시작 불필요) |

> ⚠️ 이 노트북은 **학습을 하지 않습니다.** 모델을 고르기 위한 zero-shot 측정 전용입니다.


---
## 0. 환경 설치

베이스라인과 동일하게 transformers를 소스에서 설치합니다 (Qwen3-VL은 4.57+ 필요).
**실행 후 `런타임 - 세션 다시 시작`을 한 번 하세요.**


In [ ]:
!pip -q install git+https://github.com/huggingface/transformers accelerate
!pip -q install "peft>=0.13.2" "bitsandbytes==0.46.1" datasets pillow pandas --upgrade
!pip -q install scikit-learn matplotlib tqdm --upgrade

# matplotlib 한글 폰트 (Colab 기본 이미지에 한글 폰트 없음)
!apt-get -qq install fonts-nanum > /dev/null 2>&1
print("설치 완료 → 런타임 - 세션 다시 시작")

---
## 1. Drive 마운트 & 데이터 압축 해제

베이스라인과 같은 경로를 씁니다. 이미 풀려 있으면 건너뜁니다.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

MOUNT_PATH = "/content/drive/MyDrive/ssafy"
ZIP_PATH   = f"{MOUNT_PATH}/ssafy-16-1-ai.zip"

In [ ]:
import os
from pathlib import Path

if not Path("/content/train.csv").exists():
    print("압축 해제 중... (몇 분 소요)")
    !unzip -q "{ZIP_PATH}" -d "/content/"
else:
    print("이미 압축 해제됨 — 건너뜀")

for p in ["train.csv", "test.csv", "sample_submission.csv", "train", "test"]:
    f = Path("/content") / p
    print(f"{'OK ' if f.exists() else '없음'} /content/{p}")

---
## 2. 환경 점검 & dtype 자동 선택

**T4(Turing, compute capability 7.5)는 bfloat16을 지원하지 않습니다.**
bf16을 쓰면 소프트웨어 에뮬레이션으로 느려지거나 수치가 깨집니다. GPU를 보고 자동으로 고릅니다.


In [ ]:
import gc, json, math, random, re, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
Image.MAX_IMAGE_PIXELS = None

import transformers
print("transformers :", transformers.__version__)
print("torch        :", torch.__version__)

assert torch.cuda.is_available(), "런타임 - 런타임 유형 변경 - GPU 를 선택하세요"

CC        = torch.cuda.get_device_capability(0)
GPU_NAME  = torch.cuda.get_device_name(0)
VRAM_GB   = torch.cuda.get_device_properties(0).total_memory / 1024**3
SUPPORTS_BF16 = CC[0] >= 8                      # Ampere(8.x) 이상만 bf16 네이티브
DTYPE     = torch.bfloat16 if SUPPORTS_BF16 else torch.float16

print(f"\nGPU          : {GPU_NAME}  (compute capability {CC[0]}.{CC[1]})")
print(f"VRAM         : {VRAM_GB:.1f} GB")
print(f"bf16 네이티브 : {SUPPORTS_BF16}  →  dtype = {str(DTYPE).split('.')[-1]}")

_tv = tuple(int(x) for x in transformers.__version__.split(".")[:2])
if _tv < (4, 57):
    print("\n[!] Qwen3-VL은 transformers>=4.57 필요. 0번 셀 실행 후 세션을 다시 시작하세요.")

---
## 3. 설정

`N_VAL`이 이 노트북의 시간을 결정합니다. T4에서 300장이면 모델당 대략 6~12분입니다.
300장이면 정확도의 표준오차가 약 ±2.5%p이므로 **5%p 이내 차이는 우열을 가리지 못합니다.**
시간이 남으면 `N_VAL`을 올려 판정폭을 좁히세요 (500장 → ±1.9%p).


In [ ]:
DATA_DIR   = Path("/content")                       # 압축 푼 위치
OUT_DIR    = Path(MOUNT_PATH) / "bakeoff_out"        # ★ Drive에 저장 (세션 끊겨도 생존)
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED       = 42
N_VAL      = 300        # 베이크오프 표본 수
VAL_FRAC   = 0.10       # train에서 떼어낼 검증 비율
IMAGE_SIZE = 384        # 베이스라인과 동일 조건 (해상도 실험은 승자 확정 후)
MAX_LEN    = 4096
USE_4BIT   = False      # VRAM 부족하면 True (T4 15GB에서 3~4B는 fp16으로 들어감)
FORCE_RERUN = False     # True면 저장된 결과를 무시하고 전부 다시 측정

# VRAM 기준 배치 크기 자동 결정 — OOM이 나면 아래 score_rows가 알아서 쪼갭니다
BATCH_SIZE = 4 if VRAM_GB >= 22 else (2 if VRAM_GB >= 14 else 1)

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

MODELS = [
    {"key": "qwen3vl_4b",  "id": "Qwen/Qwen3-VL-4B-Instruct",          "adapter": "qwen",   "label": "Qwen3-VL-4B"},
    {"key": "kanana_v_3b", "id": "kakaocorp/kanana-1.5-v-3b-instruct", "adapter": "kanana", "label": "Kanana-1.5-V-3B"},
    {"key": "qwen25vl_3b", "id": "Qwen/Qwen2.5-VL-3B-Instruct",        "adapter": "qwen",   "label": "Qwen2.5-VL-3B"},
]

LETTERS = ["a", "b", "c", "d"]

print(f"출력 경로   : {OUT_DIR}")
print(f"배치 크기   : {BATCH_SIZE}  (VRAM {VRAM_GB:.0f}GB 기준 자동)")
print(f"4bit 양자화 : {USE_4BIT}")

---
## 4. 데이터 로드 & 자체 val split

**`dev.csv`는 쓰지 않습니다.** dev는 개수 문제가 71.5%인데 test는 33.9%로 분포가 다르고,
5개 답안 컬럼의 만장일치가 4,413건 중 1건뿐이라 라벨도 노이즈합니다. 검증은 train에서 직접 떼어냅니다.


In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")
print("train:", train_df.shape, "| test:", test_df.shape)   # 베이스라인의 sample(n=200) 제거

def qtype(q: str) -> str:
    q = str(q)
    if "몇" in q:   return "개수"
    if "재질" in q: return "재질"
    if "색" in q:   return "색상"
    return "기타"

for df in (train_df, test_df):
    df["qtype"] = df["question"].map(qtype)

# --- 이미지 단위 split (이미지 1장당 문항 1개이므로 행 단위와 동일) ---
rng  = np.random.RandomState(SEED)
perm = rng.permutation(len(train_df))
n_val = int(len(train_df) * VAL_FRAC)
val_full = train_df.iloc[perm[:n_val]].reset_index(drop=True)
fit_df   = train_df.iloc[perm[n_val:]].reset_index(drop=True)

# 베이크오프용 서브샘플: 질문 유형 비율을 val 전체와 맞춰 층화 추출
_parts = []
for qt, g in val_full.groupby("qtype"):
    k = max(1, min(len(g), round(N_VAL * len(g) / len(val_full))))
    _parts.append(g.sample(k, random_state=SEED))
val_df = pd.concat(_parts).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print(f"\nfit {len(fit_df)} / val_full {len(val_full)} / 베이크오프 val {len(val_df)}")
print("\n질문 유형 비율 (%)")
print(pd.DataFrame({
    "test":     test_df.qtype.value_counts(normalize=True).mul(100).round(1),
    "val_full": val_full.qtype.value_counts(normalize=True).mul(100).round(1),
    "val(샘플)": val_df.qtype.value_counts(normalize=True).mul(100).round(1),
}).fillna(0))

---
## 5. 텍스트 전용 기준선 — 오늘의 합격선

이미지를 **전혀 주지 않고** 질문+선택지 텍스트만으로 학습한 분류기입니다.
GPU가 필요 없고 1분이면 끝납니다. 여기서 나온 숫자가 모든 VLM의 합격선입니다.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def to_pairs(df):
    """(문항 id, '질문 [SEP] 선택지', 정답여부) — 선택지 4개가 각각 한 행."""
    rows = []
    for gid, r in df.iterrows():
        for L in LETTERS:
            rows.append((gid, f"{r['question']} [SEP] {r[L]}",
                         int(str(r["answer"]).strip().lower() == L)))
    return pd.DataFrame(rows, columns=["gid", "text", "y"])

p_fit, p_val = to_pairs(fit_df), to_pairs(val_df)

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=2, max_features=200_000)
Xf, Xv = vec.fit_transform(p_fit.text), vec.transform(p_val.text)
txt_clf = LogisticRegression(max_iter=2000, C=2.0).fit(Xf, p_fit.y)

# to_pairs가 (문항 × a,b,c,d) 순서로 쌓으므로 그대로 (N, 4)로 접기
probs_mat = txt_clf.predict_proba(Xv)[:, 1].reshape(len(val_df), 4)
text_only_probs = probs_mat / probs_mat.sum(1, keepdims=True)
text_only_pred  = np.array([LETTERS[i] for i in text_only_probs.argmax(1)])

gold = val_df["answer"].str.strip().str.lower().values
TEXT_ONLY_ACC = float((text_only_pred == gold).mean())

np.save(OUT_DIR / "text_only_probs.npy", text_only_probs)   # 나중에 블렌딩용
print(f"텍스트 전용 정확도 : {TEXT_ONLY_ACC:.1%}   ← 모든 VLM의 합격선")
print(f"랜덤              : 25.0%")

---
## 6. 프롬프트

베이스라인과 동일하게 유지합니다. **모델 비교에서는 프롬프트를 변수로 두지 않습니다** —
한 번에 하나만 바꿔야 원인을 추적할 수 있습니다.


In [ ]:
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

def build_mc_prompt(row) -> str:
    return (
        f"{row['question']}\n"
        f"(a) {row['a']}\n(b) {row['b']}\n(c) {row['c']}\n(d) {row['d']}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

def load_image(row) -> Image.Image:
    img = Image.open(DATA_DIR / row["path"]).convert("RGB")
    img.thumbnail((IMAGE_SIZE * 2, IMAGE_SIZE * 2), Image.BICUBIC)  # 과대 이미지 사전 축소
    return img

print(build_mc_prompt(val_df.iloc[0]))
print("\n이미지 로드 테스트:", load_image(val_df.iloc[0]).size)

---
## 7. 모델 어댑터

Qwen 계열과 Kanana는 입력 API가 완전히 다릅니다.

- **Qwen**: `apply_chat_template` → `processor(text=..., images=...)`
- **Kanana**: `<image>` 플레이스홀더 + `processor.batch_encode_collate(...)`

두 방식을 같은 인터페이스(`load` / `encode` / `free`)로 감싸서 뒤쪽 평가 루프를 공용화합니다.

**kanana가 세 번 깨진 곳을 여기서 전부 막습니다** (`kanana_bakeoff.py`에서 검증한 대응):

1. `auto_map`에서 실제 클래스를 직접 로드 — Auto 클래스 추측 금지
2. 누락 패키지(`timm`, `einops`)를 에러 메시지에서 파싱해 자동 설치 후 재시도
3. `no_flash_attention` — transformers 진입점을 잠시 패치해 flash 요청을 sdpa/eager로 치환

> **왼쪽 패딩이 필수입니다.** 로짓 스코어링은 마지막 토큰 위치(`logits[:, -1, :]`)를 읽는데,
> 오른쪽 패딩이면 그 자리가 패딩 토큰이 되어 정확도가 무너집니다.


In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig
from contextlib import contextmanager
import subprocess, importlib, sys, inspect

# ═════════════════════════════════════════════════════════════════════════
# kanana 대응 (실제로 세 번 깨진 곳을 순서대로 막습니다)
# ═════════════════════════════════════════════════════════════════════════
_DEP_RE = re.compile(r"not found in your environment:\s*([^\.\n]+)")
_MOD_RE = re.compile(r"No module named ['\"]([^'\"]+)['\"]")
_PIP_ALIAS = {"cv2": "opencv-python-headless", "PIL": "pillow", "sklearn": "scikit-learn"}
_FLASH = {"flash_attention_2", "flash_attention_3"}


def _missing_packages(msg):
    """에러 메시지에서 누락 패키지 이름을 뽑습니다. flash_attn은 제외."""
    m = _DEP_RE.search(msg)
    if m:
        pkgs = [p.strip() for p in m.group(1).split(",") if p.strip()]
    else:
        m2 = _MOD_RE.search(msg)
        pkgs = [m2.group(1).split(".")[0]] if m2 else []
    return [_PIP_ALIAS.get(p, p) for p in pkgs if p != "flash_attn"]


def _pip_install(pkgs):
    print(f"[deps] 누락 패키지 자동 설치: {pkgs}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
    importlib.invalidate_caches()


def _clear_remote_modules():
    """실패한 remote code 모듈이 sys.modules에 반쯤 남으면 재시도가 계속 깨집니다."""
    for k in [k for k in list(sys.modules) if k.startswith("transformers_modules")]:
        del sys.modules[k]


def _scrub_flash_attn(cfg, impl):
    """config와 모든 하위 config에서 flash_attention_2를 걷어냅니다."""
    seen = set()

    def walk(c):
        if c is None or id(c) in seen or not hasattr(c, "to_dict"):
            return
        seen.add(id(c))
        for a in ("_attn_implementation", "attn_implementation"):
            if getattr(c, a, None) in _FLASH:
                try: setattr(c, a, impl)
                except Exception: pass
        try: c._attn_implementation = impl
        except Exception: pass
        for v in list(vars(c).values()):
            if hasattr(v, "to_dict"):
                walk(v)

    walk(cfg)
    return cfg


@contextmanager
def no_flash_attention(impl="eager", verbose=True):
    """
    kanana의 modeling.py는 vision tower를 이렇게 하드코딩합니다:

        try:
            self.vision_model = CustomQwen2VLVE._from_config(
                config.vision_config, attn_implementation="flash_attention_2")
        except Exception:
            self.vision_model = CustomQwen2VLVE._from_config(config.vision_config)

    try/except가 있는데도 터지는 이유: 1차 호출이 실패하면서 transformers가
    config.vision_config._attn_implementation 을 'flash_attention_2'로
    **이미 써버립니다**. 폴백 호출이 그 오염된 config를 그대로 쓰니 또 터지고,
    이번엔 잡아주는 곳이 없어 ImportError가 밖으로 나옵니다.
    (config를 미리 청소해도 1차 호출이 다시 오염시키므로 소용없습니다.)

    → transformers 진입점에서 flash 요청 자체를 impl로 바꿔치기합니다.
      블록을 벗어나면 원상 복구합니다.

    flash_attn 설치는 해결책이 아닙니다: Ampere(sm80) 이상 전용이고
    빌드에만 10분 이상 걸립니다.
    """
    from transformers.modeling_utils import PreTrainedModel as PM
    NAMES = ("_from_config",                            # 모든 버전
             "_check_and_adjust_attn_implementation",   # transformers 4.5x+
             "_autoset_attn_implementation")            # 구버전
    saved, patched = {}, []

    def coerce(a, kw):
        a = [impl if (isinstance(v, str) and v in _FLASH) else v for v in a]
        for k, v in list(kw.items()):
            if isinstance(v, str) and v in _FLASH: kw[k] = impl
            elif k == "use_flash_attention_2":     kw[k] = False
        for v in list(a) + list(kw.values()):
            if hasattr(v, "to_dict"): _scrub_flash_attn(v, impl)
        return a, kw

    for name in NAMES:
        raw = inspect.getattr_static(PM, name, None)
        if raw is None:
            continue
        saved[name] = (raw, name in PM.__dict__)
        is_cm, is_sm = isinstance(raw, classmethod), isinstance(raw, staticmethod)
        fn = raw.__func__ if (is_cm or is_sm) else raw

        def make(fn=fn):
            def wrapper(first, *a, **kw):
                a, kw = coerce(list(a), kw)
                return fn(first, *a, **kw)
            return wrapper

        w = make()
        setattr(PM, name, classmethod(w) if is_cm else (staticmethod(w) if is_sm else w))
        patched.append(name)

    if verbose and patched:
        print(f"[flash] attn_implementation={impl} 로 강제 (패치: {', '.join(patched)})")
    try:
        yield
    finally:
        for name, (raw, own) in saved.items():
            if own:
                setattr(PM, name, raw)
            else:
                try: delattr(PM, name)
                except AttributeError: pass


def _resolve_model_class(model_id, verbose=True):
    """
    모델을 어떤 클래스로 열어야 하는지 config.auto_map에서 직접 읽습니다.

    kanana는 auto_map에 AutoModelForVision2Seq 로만 등록돼 있어서
    AutoModelForImageTextToText 로 열면 'Unrecognized configuration class'가 납니다.
    게다가 transformers 버전에 따라 Vision2Seq가 ImageTextToText의 별칭이거나
    아예 없어서, Auto 클래스 이름만으로는 못 맞춥니다
    → auto_map이 있으면 동적 모듈에서 실제 클래스를 직접 가져옵니다.
    """
    import transformers
    from transformers import AutoConfig
    KEYS = ("AutoModelForVision2Seq", "AutoModelForImageTextToText",
            "AutoModelForCausalLM", "AutoModel")
    try:
        cfg = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
        amap = getattr(cfg, "auto_map", None) or {}
    except Exception as e:
        if verbose: print(f"[cls] config 읽기 실패({type(e).__name__}) → 표준 Auto 클래스")
        amap = {}

    if amap:
        ref = next((amap[k] for k in KEYS if k in amap), None)
        if ref:
            try:
                from transformers.dynamic_module_utils import get_class_from_dynamic_module
                C = get_class_from_dynamic_module(ref, model_id)
                if verbose: print(f"[cls] auto_map 직접 로드 → {ref}")
                return C
            except Exception as e:
                if verbose: print(f"[cls] 동적 로드 실패({type(e).__name__}) → Auto 폴백")
        for k in KEYS:
            if k in amap and getattr(transformers, k, None) is not None:
                if verbose: print(f"[cls] auto_map → {k}")
                return getattr(transformers, k)

    for k in ("AutoModelForImageTextToText", "AutoModelForVision2Seq"):
        C = getattr(transformers, k, None)
        if C is not None:
            if verbose: print(f"[cls] {k}")
            return C
    raise RuntimeError("사용 가능한 모델 클래스를 찾지 못했습니다")


def _load_kwargs():
    kw = {"device_map": "auto", "trust_remote_code": True}
    if USE_4BIT:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE)
    else:
        kw["dtype"] = DTYPE
    return kw


class BaseAdapter:
    ATTN_ORDER = ("sdpa", "eager")

    def __init__(self, spec):
        self.spec = spec; self.model = None; self.processor = None

    def _from_pretrained(self, max_dep_retry=4):
        import copy as _copy
        from transformers import AutoConfig
        mid = self.spec["id"]

        for _ in range(max_dep_retry):
            try:
                _clear_remote_modules()
                Cls = _resolve_model_class(mid)
                base_cfg = AutoConfig.from_pretrained(mid, trust_remote_code=True)

                last = None
                for impl in self.ATTN_ORDER:
                    cfg = _scrub_flash_attn(_copy.deepcopy(base_cfg), impl)
                    base_kw = dict(_load_kwargs(), config=cfg)
                    variants = []
                    for extra in ({"attn_implementation": impl}, {}):
                        for dt in ("dtype", "torch_dtype"):
                            kw = dict(base_kw, **extra)
                            if "dtype" in kw and dt == "torch_dtype":
                                kw["torch_dtype"] = kw.pop("dtype")
                            variants.append(kw)

                    with no_flash_attention(impl, verbose=False):   # ★ 핵심
                        for kw in variants:
                            try:
                                m = Cls.from_pretrained(mid, **kw)
                                print(f"[load] attn_implementation={impl}")
                                return m
                            except TypeError as e:
                                last = e; continue
                            except ImportError:
                                raise
                            except Exception as e:
                                last = e
                                print(f"[load] attn={impl} 실패: {type(e).__name__}: {str(e)[:150]}")
                                gc.collect(); torch.cuda.empty_cache()
                                break
                raise last if last else RuntimeError("모델 로드 실패")

            except ImportError as e:
                msg = str(e)
                if "flash_attn" in msg or "FlashAttention" in msg:
                    raise RuntimeError(
                        "flash_attention_2 요청을 끝내 우회하지 못했습니다. "
                        "BaseAdapter.ATTN_ORDER = ('eager',) 로 고정해 보세요. "
                        "flash_attn 설치는 해결책이 아닙니다 (sm80 미만 미지원)."
                    ) from e
                pkgs = _missing_packages(msg)
                if not pkgs:
                    raise
                _pip_install(pkgs)
        raise RuntimeError("의존성 자동 설치를 반복했지만 로드에 실패했습니다")

    def free(self):
        self.model = None; self.processor = None
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()


class QwenAdapter(BaseAdapter):
    def load(self):
        px = IMAGE_SIZE * IMAGE_SIZE
        try:
            self.processor = AutoProcessor.from_pretrained(
                self.spec["id"], min_pixels=px, max_pixels=px, trust_remote_code=True)
        except TypeError:
            self.processor = AutoProcessor.from_pretrained(self.spec["id"], trust_remote_code=True)
        self.processor.tokenizer.padding_side = "left"      # ★ 로짓 스코어링에 필수
        self.model = self._from_pretrained(); self.model.eval()
        return self

    def encode(self, rows):
        texts, images = [], []
        for r in rows:
            messages = [
                {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
                {"role": "user",   "content": [{"type": "image"},
                                               {"type": "text", "text": build_mc_prompt(r)}]},
            ]
            texts.append(self.processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True))
            images.append(load_image(r))
        return self.processor(text=texts, images=images, return_tensors="pt",
                              padding=True, truncation=True, max_length=MAX_LEN)


class KananaAdapter(BaseAdapter):
    def load(self):
        self.processor = AutoProcessor.from_pretrained(self.spec["id"], trust_remote_code=True)
        self.model = self._from_pretrained(); self.model.eval()
        return self

    def encode(self, rows):
        batch = [{
            "image": [load_image(r)],
            "conv": [{"role": "user", "content": "<image>"},
                     {"role": "user", "content": SYSTEM_INSTRUCT + "\n\n" + build_mc_prompt(r)}],
        } for r in rows]
        return self.processor.batch_encode_collate(
            batch, padding_side="left", add_generation_prompt=True, max_length=MAX_LEN)


ADAPTERS = {"qwen": QwenAdapter, "kanana": KananaAdapter}
print("어댑터 준비 완료:", list(ADAPTERS))

---
## 8. 로짓 비교 추론 + OOM 자동 대응

`generate()`를 쓰지 않습니다. 정답이 한 글자이므로 **forward 한 번**으로 다음 토큰 분포에서
a/b/c/d 네 개만 꺼내 비교하면 됩니다. 파싱 실패가 0이 되고 2~3배 빨라지며,
확률값이 남아서 나중에 텍스트 사전확률과 블렌딩할 수 있습니다.

`score_rows`는 OOM이 나면 배치를 **재귀적으로 반씩 쪼개** 다시 시도합니다.
T4에서 큰 이미지가 몰린 배치를 만나도 죽지 않습니다.

> **모델을 바꿀 때마다 아래 검증이 통과하는지 확인하세요.** 토크나이저에 따라 `"a"`와 `" a"`가
> 갈리는데, 잘못된 id를 잡으면 정확도가 조용히 25% 근처에 고정됩니다.


In [ ]:
def get_letter_ids(tokenizer, verbose=True):
    """a/b/c/d 각각의 단일 토큰 id를 찾고, 서로 다른지 검증."""
    ids, report = [], []
    for L in LETTERS:
        enc = tokenizer.encode(L, add_special_tokens=False)
        if not enc:
            enc = tokenizer.encode(" " + L, add_special_tokens=False)
        ids.append(enc[0])
        report.append((L, enc, repr(tokenizer.decode([enc[0]]))))
    if verbose:
        print(f"{'letter':<9}{'ids':<16}{'decode(id[0])'}")
        for L, e, d in report:
            print(f"{L:<9}{str(e):<16}{d}")
    assert len(set(ids)) == 4, f"[FAIL] LETTER_IDS 중복! {ids} — 토크나이저 확인 필요"
    if verbose: print(f"[OK] 4개 id가 모두 다름: {ids}")
    return ids


@torch.no_grad()
def _forward_probs(model, inputs, letter_ids):
    dev = next(model.parameters()).device
    inputs = {k: (v.to(dev) if torch.is_tensor(v) else v) for k, v in inputs.items()}
    try:
        out = model(**inputs)
    except TypeError:   # generate 전용 키가 섞인 경우 제거 후 재시도
        drop = {"attention_mask_2d", "generation_config", "max_length", "padding_side"}
        out = model(**{k: v for k, v in inputs.items() if k not in drop})
    logits = out.logits[:, -1, :].float()          # fp16 오버플로 방지로 float 승격
    return logits[:, letter_ids].softmax(-1).cpu().numpy()


def score_rows(adapter, rows, letter_ids):
    """OOM이면 배치를 반으로 쪼개 재귀 재시도."""
    try:
        return _forward_probs(adapter.model, adapter.encode(rows), letter_ids)
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        if len(rows) == 1:
            raise
        mid = len(rows) // 2
        return np.concatenate([score_rows(adapter, rows[:mid], letter_ids),
                               score_rows(adapter, rows[mid:], letter_ids)], 0)

print("로짓 스코어링 준비 완료")

---
## 9. 단일 모델 평가 (결과는 Drive에 즉시 저장)

모델 하나를 로드 → val 평가 → **결과 저장 → VRAM 반납**. 세션이 끊겨도
다시 실행하면 이미 끝난 모델은 건너뜁니다.


In [ ]:
def result_path(key): return OUT_DIR / f"result_{key}.json"


def evaluate_model(spec, df):
    print(f"\n{'='*66}\n▶ {spec['label']}  ({spec['id']})\n{'='*66}")
    t_load = time.time()
    adapter = ADAPTERS[spec["adapter"]](spec).load()
    print(f"로드 완료 ({time.time()-t_load:.0f}s) · VRAM {torch.cuda.memory_allocated()/1024**3:.1f}GB")

    letter_ids = get_letter_ids(adapter.processor.tokenizer)

    all_probs, t0 = [], time.time()
    for s in tqdm(range(0, len(df), BATCH_SIZE), desc=spec["label"], unit="batch"):
        rows = [df.iloc[i] for i in range(s, min(s + BATCH_SIZE, len(df)))]
        all_probs.append(score_rows(adapter, rows, letter_ids))
    elapsed = time.time() - t0

    probs = np.concatenate(all_probs, 0)
    pred  = np.array([LETTERS[i] for i in probs.argmax(1)])
    g     = df["answer"].str.strip().str.lower().values
    acc   = float((pred == g).mean())

    by_type = {}
    for qt in df.qtype.unique():
        m = (df.qtype == qt).values
        by_type[qt] = float((pred[m] == g[m]).mean())

    res = {"key": spec["key"], "label": spec["label"], "id": spec["id"],
           "acc": acc, "by_type": by_type,
           "pred_dist": {L: float((pred == L).mean()) for L in LETTERS},
           "sec_per_sample": elapsed / len(df),
           "n_val": len(df), "gpu": GPU_NAME, "dtype": str(DTYPE).split(".")[-1],
           "use_4bit": USE_4BIT, "image_size": IMAGE_SIZE,
           "probs": probs.tolist()}

    with open(result_path(spec["key"]), "w", encoding="utf-8") as f:   # ★ 즉시 Drive 저장
        json.dump(res, f, ensure_ascii=False)

    adapter.free()
    print(f"\n정확도 {acc:.1%}  (텍스트 전용 {TEXT_ONLY_ACC:.1%} / 랜덤 25.0%)"
          f"  ·  {res['sec_per_sample']:.2f}s/샘플"
          f"  ·  test 5,074장 추정 {res['sec_per_sample']*5074/60:.0f}분")
    return res

---
## 10. 베이크오프 실행

모델 3개를 순차 로드/반납합니다. 다운로드 포함 첫 실행은 모델당 10~20분 걸릴 수 있습니다.
**세션이 끊기면 이 셀만 다시 실행하세요** — 끝난 모델은 Drive에서 불러옵니다.


In [ ]:
results, failures = [], []

for spec in MODELS:
    rp = result_path(spec["key"])
    if rp.exists() and not FORCE_RERUN:
        results.append(json.load(open(rp, encoding="utf-8")))
        print(f"[SKIP] {spec['label']} — 저장된 결과 사용 ({results[-1]['acc']:.1%})")
        continue
    try:
        results.append(evaluate_model(spec, val_df))
    except Exception as e:
        failures.append((spec["label"], f"{type(e).__name__}: {e}"))
        print(f"\n[FAIL] {spec['label']} → {type(e).__name__}: {e}")
        gc.collect(); torch.cuda.empty_cache()

results.sort(key=lambda r: [m["key"] for m in MODELS].index(r["key"]))

if failures:
    print("\n실패한 모델:")
    for lbl, err in failures: print(f"  - {lbl}: {err}")

pd.DataFrame([{
    "모델": r["label"], "정확도": f"{r['acc']:.1%}",
    "vs 텍스트전용": f"{r['acc']-TEXT_ONLY_ACC:+.1%}",
    "s/샘플": f"{r['sec_per_sample']:.2f}",
    "test 5074장": f"{r['sec_per_sample']*5074/60:.0f}분",
} for r in sorted(results, key=lambda r: -r["acc"])])

---
## 11. 시각화

네 개의 패널:

1. **전체 정확도** — 합격선(텍스트 전용)과 랜덤을 기준선으로 함께 표시
2. **질문 유형별 정확도** — 개수 세기에서 갈리는지 확인 (test의 34%)
3. **예측 분포 진단** — 한 글자로 쏠리면 프롬프트나 토큰 id가 잘못된 신호
4. **추론 속도** — 하루 예산에서 학습 시간이 얼마나 남는지 결정


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import ft2font
from matplotlib.ticker import PercentFormatter

# ─────────────────────────────────────────────────────────────────────
# 한글 폰트 설정
#
# fontManager.ttflist는 캐시 기반이라 오래되면 시스템 폰트를 놓칩니다
# (Windows에서 맑은고딕이 안 잡히는 전형적인 원인).
# 그래서 (1) 폰트 파일 경로를 직접 훑고, (2) ft2font로 '한' 글리프가
# 실제로 있는지 검증한 뒤, (3) addfont로 등록합니다.
# 끝내 못 찾으면 그래프 라벨을 영어로 자동 전환합니다 (□ 방지).
# ─────────────────────────────────────────────────────────────────────
_KNOWN_PATHS = [
    # Windows
    r"C:\Windows\Fonts\malgun.ttf", r"C:\Windows\Fonts\malgunbd.ttf",
    r"C:\Windows\Fonts\NanumGothic.ttf", r"C:\Windows\Fonts\gulim.ttc",
    r"C:\Windows\Fonts\batang.ttc",
    # macOS
    "/System/Library/Fonts/Supplemental/AppleGothic.ttf",
    "/System/Library/Fonts/AppleSDGothicNeo.ttc",
    # Linux / Colab
    "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
    "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf",
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
]

# 전수 조사에서 우선할 한글 폰트 이름 (소문자 부분일치)
_PREFER = ["malgun", "nanum", "apple sd gothic", "applegothic", "noto sans cjk",
           "noto sans kr", "source han sans", "pretendard", "spoqa", "gulim",
           "dotum", "batang", "gothic"]
# 한글 글리프는 있지만 본문용으로 부적합한 폰트 (bitmap·폴백·기호 폰트)
_AVOID = ["unifont", "sample", "symbol", "emoji", "wingding", "webding"]


def _has_hangul(path):
    try:
        return ft2font.FT2Font(path).get_char_index(ord("한")) != 0
    except Exception:
        return False


def _register(path, why=""):
    fm.fontManager.addfont(path)
    name = fm.FontProperties(fname=path).get_name()
    plt.rcParams["font.family"] = name
    print(f"[font] {name}  ({path}){why}")
    return name


def setup_korean_font():
    for p in _KNOWN_PATHS:                              # 1) 알려진 경로 우선 (빠름)
        if Path(p).exists() and _has_hangul(p):
            return _register(p)

    hits = []                                           # 2) 시스템 전수 조사 (캐시 우회)
    for ext in ("ttf", "otf"):
        for p in fm.findSystemFonts(fontext=ext):
            if not _has_hangul(p):
                continue
            try:
                name = fm.FontProperties(fname=p).get_name()
            except Exception:
                continue
            low = (name + " " + Path(p).name).lower()
            if any(bad in low for bad in _AVOID):
                rank = 2                                # 최후의 수단
            elif any(good in low for good in _PREFER):
                rank = 0                                # 제대로 된 한글 폰트
            else:
                rank = 1
            hits.append((rank, name, p))

    if hits:
        hits.sort(key=lambda h: h[0])
        rank, name, p = hits[0]
        return _register(p, "  [!] 본문용 한글 폰트가 아닐 수 있습니다" if rank == 2 else "")
    return None


KO_FONT = setup_korean_font()
KO = KO_FONT is not None
plt.rcParams["axes.unicode_minus"] = False
if not KO:
    print("[font] 한글 폰트를 찾지 못해 그래프 라벨을 영어로 표시합니다.")
    print("       Windows: 보통 C:\\Windows\\Fonts\\malgun.ttf 가 있습니다.")
    print("       Linux/Colab: !apt-get install -y fonts-nanum 후 이 셀만 다시 실행하세요.")


def T(ko, en):
    """한글 폰트가 있으면 한글, 없으면 영어."""
    return ko if KO else en


QT_EN = {"개수": "Count", "재질": "Material", "색상": "Color", "기타": "Other"}

SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e5e4e0"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]          # 검증된 카테고리 팔레트 (3슬롯)
CMAP = {r["label"]: SERIES[i] for i, r in enumerate(results)}

def tidy(ax):
    ax.set_facecolor(SURFACE)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    for s in ("left", "bottom"): ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK2, length=0, labelsize=9)
    ax.grid(axis="x", color=GRID, lw=.8, zorder=0); ax.set_axisbelow(True)

order = sorted(results, key=lambda r: r["acc"])
fig, axes = plt.subplots(2, 2, figsize=(13, 8.6), facecolor=SURFACE)
fig.suptitle(T(f"Zero-shot 모델 베이크오프  ·  val {len(val_df)}장  ·  {IMAGE_SIZE}px  ·  {GPU_NAME} / {str(DTYPE).split('.')[-1]}  ·  학습 없음",
               f"Zero-shot model bakeoff  ·  val {len(val_df)}  ·  {IMAGE_SIZE}px  ·  {GPU_NAME} / {str(DTYPE).split('.')[-1]}  ·  no training"),
             fontsize=13, fontweight="bold", color=INK, x=.5, y=.985)

# ── 1. 전체 정확도 ────────────────────────────────────────────────
ax = axes[0, 0]; tidy(ax)
y = np.arange(len(order))
ax.barh(y, [r["acc"] for r in order], height=.42,
        color=[CMAP[r["label"]] for r in order], zorder=3)
for i, r in enumerate(order):                                   # 직접 라벨 (대비 완화 규칙)
    ax.text(r["acc"] + .012, i, f"{r['acc']:.1%}", va="center",
            fontsize=10.5, fontweight="bold", color=INK)
ax.axvline(.25, color=INK2, ls=":", lw=1.4, zorder=4)
ax.axvline(TEXT_ONLY_ACC, color="#e34948", ls="--", lw=1.8, zorder=4)
ax.text(.25, len(order) - .3, T(" 랜덤 25%", " random 25%"), fontsize=8.5, color=INK2, va="bottom")
ax.text(TEXT_ONLY_ACC, len(order) - .3,
        T(f" 텍스트 전용 {TEXT_ONLY_ACC:.0%} ← 합격선", f" text-only {TEXT_ONLY_ACC:.0%} ← pass line"),
        fontsize=8.5, color="#e34948", va="bottom", fontweight="bold")
ax.set_yticks(y); ax.set_yticklabels([r["label"] for r in order], fontsize=10, color=INK)
ax.set_xlim(0, max(.75, max(r["acc"] for r in results) + .14))
ax.set_ylim(-.6, len(order) - .1)
ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.set_title(T("전체 정확도", "Overall accuracy"), fontsize=11,
             fontweight="bold", color=INK, loc="left", pad=10)

# ── 2. 질문 유형별 ────────────────────────────────────────────────
ax = axes[0, 1]; tidy(ax)
types = [t for t in ["개수", "재질", "색상", "기타"] if t in val_df.qtype.unique()]
share = val_df.qtype.value_counts(normalize=True)
x = np.arange(len(types)); w = .8 / len(results)
for i, r in enumerate(results):
    vals = [r["by_type"].get(t, np.nan) for t in types]
    xs = x + i * w - .4 + w / 2
    ax.bar(xs, vals, w * .88, label=r["label"], color=CMAP[r["label"]], zorder=3)
    for xx, v in zip(xs, vals):
        if not np.isnan(v):
            ax.text(xx, v + .015, f"{v:.0%}", ha="center", fontsize=7.5, color=INK2)
ax.axhline(TEXT_ONLY_ACC, color="#e34948", ls="--", lw=1.4, zorder=4)
ax.set_xticks(x)
ax.set_xticklabels([f"{T(t, QT_EN[t])}\n({share.get(t,0):.0%})" for t in types],
                   fontsize=9.5, color=INK)
ax.set_ylim(0, 1.26)                       # 범례가 막대를 가리지 않도록 여유
ax.set_yticks([0, .2, .4, .6, .8, 1.0])
ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.grid(axis="y", color=GRID, lw=.8, zorder=0); ax.grid(axis="x", visible=False)
ax.legend(fontsize=8.5, frameon=False, loc="upper right", ncol=3,
          columnspacing=1.1, handlelength=1.2)
ax.set_title(T("질문 유형별 정확도  (괄호 = val 내 비중)",
                "Accuracy by question type  (share of val)"),
             fontsize=11, fontweight="bold", color=INK, loc="left", pad=10)

# ── 3. 예측 분포 진단 ─────────────────────────────────────────────
ax = axes[1, 0]; tidy(ax)
x = np.arange(4); w = .8 / len(results)
for i, r in enumerate(results):
    ax.bar(x + i * w - .4 + w / 2, [r["pred_dist"][L] for L in LETTERS], w * .88,
           label=r["label"], color=CMAP[r["label"]], zorder=3)
ax.axhline(.25, color=INK2, ls=":", lw=1.4, zorder=4)
ax.text(3.45, .255, T("균등 25%", "uniform 25%"), fontsize=8, color=INK2, ha="right")
ax.set_xticks(x); ax.set_xticklabels([f"({L})" for L in LETTERS], fontsize=10, color=INK)
ax.set_ylim(0, max(.42, max(max(r["pred_dist"].values()) for r in results) * 1.25))
ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.grid(axis="y", color=GRID, lw=.8, zorder=0); ax.grid(axis="x", visible=False)
ax.legend(fontsize=8.5, frameon=False, loc="upper right")
ax.set_title(T("예측 분포 진단  ·  한 글자로 쏠리면 프롬프트/토큰 id 문제",
                "Prediction distribution  ·  a skew to one letter = prompt/token-id bug"),
             fontsize=11, fontweight="bold", color=INK, loc="left", pad=10)

# ── 4. 추론 속도 ─────────────────────────────────────────────────
ax = axes[1, 1]; tidy(ax)
sp = sorted(results, key=lambda r: -r["sec_per_sample"])
y = np.arange(len(sp)); mins = [r["sec_per_sample"] * 5074 / 60 for r in sp]
ax.barh(y, mins, height=.42, color=[CMAP[r["label"]] for r in sp], zorder=3)
for i, (r, m) in enumerate(zip(sp, mins)):
    ax.text(m + max(mins) * .02, i, T(f"{m:.0f}분  ({r['sec_per_sample']:.2f}s/장)",
                                      f"{m:.0f} min  ({r['sec_per_sample']:.2f}s/img)"),
            va="center", fontsize=9.5, color=INK)
ax.set_yticks(y); ax.set_yticklabels([r["label"] for r in sp], fontsize=10, color=INK)
ax.set_xlim(0, max(mins) * 1.42); ax.set_ylim(-.6, len(sp) - .1)
ax.set_xlabel(T("test 5,074장 추론 예상 시간 (분)",
                "estimated inference time for 5,074 test images (min)"), fontsize=9, color=INK2)
ax.set_title(T("추론 비용  ·  남는 시간이 곧 학습 예산",
                "Inference cost  ·  what is left is the training budget"),
             fontsize=11, fontweight="bold", color=INK, loc="left", pad=10)

plt.tight_layout(rect=[0, 0, 1, .96])
plt.savefig(OUT_DIR / "bakeoff.png", dpi=160, facecolor=SURFACE, bbox_inches="tight")
plt.show()
print("저장:", OUT_DIR / "bakeoff.png")

---
## 12. 판정

표본이 300장이면 표준오차가 약 ±2.5%p입니다. **차이가 5%p 이내면 무승부로 취급**하고,
그럴 때는 더 빠르고 라이선스가 깨끗한 쪽(Apache 2.0인 Qwen 계열)을 고르는 것이 하루 예산에서 합리적입니다.


In [ ]:
se = math.sqrt(.25 * .75 / len(val_df))
print(f"표본 {len(val_df)}장 · 표준오차 ≈ ±{se:.1%}p · 무승부 판정폭 ≈ {2*se:.1%}p\n")

ranked = sorted(results, key=lambda r: -r["acc"])
for i, r in enumerate(ranked, 1):
    gap  = r["acc"] - TEXT_ONLY_ACC
    flag = "PASS" if gap > 2 * se else ("FAIL — 이미지 기여 없음" if gap <= 0 else "애매 — 통계적 무의미")
    print(f"{i}. {r['label']:<22} {r['acc']:.1%}   텍스트전용 대비 {gap:+.1%}p   [{flag}]")

if ranked:
    best = ranked[0]
    tied = [r for r in ranked[1:] if best["acc"] - r["acc"] < 2 * se]
    print(f"\n승자: {best['label']}  ({best['acc']:.1%})")
    if tied:
        print("무승부권:", ", ".join(r["label"] for r in tied),
              "→ 더 빠르고 라이선스가 깨끗한 쪽 선택 권장")
    if best["acc"] - TEXT_ONLY_ACC <= 2 * se:
        print("\n[!] 어떤 모델도 텍스트 전용 기준선을 유의하게 못 넘었습니다.")
        print("    학습으로 넘어가지 말고 먼저 점검하세요:")
        print("      1) get_letter_ids 출력 — id 4개가 정말 다른가")
        print("      2) 예측 분포 패널 — 한 글자로 쏠려 있지 않은가")
        print("      3) padding_side='left'가 적용됐는가")
        print("      4) T4에서 dtype이 fp16으로 잡혔는가 (bf16이면 수치가 깨질 수 있음)")
        print("      5) 이미지가 실제로 프롬프트에 들어갔는가 (load_image 출력)")
    else:
        print(f"\n다음 단계 → {best['label']}에만 남은 학습 시간 전부 투입 (Step 4)")
        print(f"          → 해상도 {IMAGE_SIZE} → 512 실험은 승자 확정 후 (Step 6 레버 1순위)")

with open(OUT_DIR / "bakeoff_summary.json", "w", encoding="utf-8") as f:
    json.dump({"n_val": len(val_df), "text_only_acc": TEXT_ONLY_ACC,
               "gpu": GPU_NAME, "dtype": str(DTYPE).split(".")[-1],
               "image_size": IMAGE_SIZE, "use_4bit": USE_4BIT,
               "ranking": [{k: r.get(k) for k in ("key", "label", "acc", "by_type",
                                                  "pred_dist", "sec_per_sample")}
                           for r in ranked],
               "failures": failures}, f, ensure_ascii=False, indent=2)
print(f"\n요약 저장: {OUT_DIR / 'bakeoff_summary.json'}")

---
### Drive에 남는 것 (`MyDrive/ssafy/bakeoff_out/`)

- `result_<모델키>.json` — 모델별 정확도·유형별·예측분포·확률 원본 (세션이 끊겨도 생존, 재실행 시 자동 재사용)
- `bakeoff_summary.json` — 순위와 실행 조건 요약
- `text_only_probs.npy` — 텍스트 전용 확률. 나중에 VLM 확률과 가중 블렌딩할 때 그대로 사용
- `bakeoff.png` — 4패널 그림

**다음**: 승자 모델로 QLoRA 학습 노트북(Step 4). 이 노트북의 어댑터·로짓 스코어링·val split을 그대로 재사용합니다.

### Colab에서 막히면

| 증상 | 대응 |
|---|---|
| `OutOfMemoryError`가 끝까지 남음 | 3번 셀에서 `USE_4BIT = True` 후 해당 모델만 재실행 |
| 세션이 끊김 | 1·2·3·4·5·6·7·8·9번 셀 다시 실행 → 10번 셀 실행 (끝난 모델은 건너뜀) |
| 한 모델만 실패 | 나머지 결과는 이미 Drive에 저장됨. 그 모델만 빼고 판정해도 됩니다 |
| 한글이 □로 나옴 | 0번 셀의 `apt-get install fonts-nanum` 실행 후 11번 셀 재실행 |
